In [1]:
# 필요한 패키지 불러오기
import pandas as pd
from selenium.webdriver.common.by import By
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
from openpyxl import Workbook
from bs4 import BeautifulSoup
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
import time
import requests
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import NoSuchElementException


In [2]:
# Webdriver headless mode setting
options = webdriver.ChromeOptions()
# options.add_argument("--headless=new") # headless 모드: 실제 창이 뜨지 않은 채 메모리상에서만 브라우저 구동
options.add_argument('window-size=1920x1080') # 브라우저의 가상 창 크기 설정: 창 크기 명시하지 않으며 요소(버튼 등)가 겹쳐서 클릭이 안되는 문제 있을 수 있음 
options.add_argument("disable-gpu") # 그래픽 카드(GPU) 가속 기능 끔. 과거 헤드리스 모드에서 GPU 가속 활성화돼 있으면 오류 발생하는 버그 있었음

# BS4 setting for secondary access
session = requests.Session()
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36"}

retries = Retry(total=5,
                backoff_factor=0.1, # 재시도 사이의 대기 시간
                status_forcelist=[500, 502, 503, 504])

session.mount('https://', HTTPAdapter(max_retries=retries))
# session.mount('http://', HTTPAdapter(max_retries=retries))

In [5]:
SELECT_KEYWORDS = [
    "분위기가 좋아요", "인테리어", "조용한", "트렌디", "데이트", "특별한 날", "가족", "가성비"
]
COMMENT_KEYWORDS = [
    "혼밥", "국물", "따뜻한", "파전", "아늑한",
    "비오는 날", "이자카야", "전골", "시원한", "여름", "냉면", "빙수",
    "브런치", "테라스", "창가", "날씨 좋은", "신선한"
]

def get_keyword_counts(restaurant_name, id, options):
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.get(f"https://m.place.naver.com/restaurant/{id}/review/visitor")
    driver.implicitly_wait(30)

    # --- 식당 카테고리 ---
    category = "미분류"
    try:
        category_el = driver.find_element(By.CLASS_NAME, '.lnJFt')
        category = category_el.text.strip()
        print(f"[{restaurant_name}] 카테고리 수집 성공: {category}")
    except NoSuchElementException:
        print(f"[{restaurant_name}] 카테고리(.lnJFt) 요소를 찾지 못했습니다.")
    except Exception as e:
        print(f"[{restaurant_name}] 카테고리 수집 중 에러 발생: {e}")

    driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.END)
    time.sleep(1.0)

    target2_clicked = False

    try:
        while True:
            driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.PAGE_DOWN)

            try:
                target = driver.find_element(By.CLASS_NAME, 'TeItc')
                driver.execute_script("arguments[0].click();", target)

            except NoSuchElementException:
                if not target2_clicked:
                    print(f"[{restaurant_name}] target을 찾지 못했습니다! target2 동작을 실행합니다.")
                    target2 = driver.find_element(By.CLASS_NAME, 'Kv9y6')
                    driver.execute_script("arguments[0].click();", target2)
                    target2_clicked = True
                else:
                    break

    except Exception as e:
        print(f"[{restaurant_name}] 에러가 발생했습니다: {e}")
        print('finish')

    # ── 키워드 카운팅 초기화 ───────────────────────
    row = {"식당명": restaurant_name, "카테고리": category}
    for keyword in SELECT_KEYWORDS + COMMENT_KEYWORDS:
        row[keyword] = 0

    # ── 더보기 버튼 한 번에 클릭 ─────────────────
    try:
        more_btns = driver.find_elements(By.CSS_SELECTOR, ".pui__vn15t2")
        print(f"[{restaurant_name}] 더보기 버튼 {len(more_btns)}개 발견")

        for btn in more_btns:
            try:
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.3)
            except Exception as e:
                print(f"[{restaurant_name}] 더보기 버튼 클릭 에러: {e}")
                continue

    except Exception as e:
        print(f"[{restaurant_name}] 더보기 버튼 에러: {e}")

    # ── SELECT_KEYWORDS 파싱 ───────────────────────
    try:
        keyword_els = driver.find_elements(By.CSS_SELECTOR, ".pui__jhpEyP")
        print(f"[{restaurant_name}] SELECT 키워드 요소 {len(keyword_els)}개 수집")

        for el in keyword_els:
            keyword_text = el.text.strip()
            for keyword in SELECT_KEYWORDS:
                if keyword in keyword_text:
                    row[keyword] += 1

    except Exception as e:
        print(f"[{restaurant_name}] SELECT_KEYWORDS 파싱 에러: {e}")

    # ── COMMENT_KEYWORDS 파싱 ─────────────────────
    try:
        comments = driver.find_elements(By.CSS_SELECTOR, ".pui__vn15t2")
        print(f"[{restaurant_name}] 댓글 {len(comments)}개 수집")

        for el in comments:
            comment_text = el.text
            for keyword in COMMENT_KEYWORDS:
                if keyword in comment_text:
                    row[keyword] += 1

    except Exception as e:
        print(f"[{restaurant_name}] COMMENT_KEYWORDS 파싱 에러: {e}")

    finally:
        driver.quit()

    return row

In [10]:
# ── 실행 ───────────────────────────────────────────
restaurants = [
    {"name": "식당A", "id": "2050506031"},
]

rows = []
for r in restaurants:
    print(f"\n[{r['name']}] 파싱 시작...")
    row = get_keyword_counts(r["name"], r["id"], options)
    rows.append(row)

df = pd.DataFrame(rows).set_index("식당명")
df.to_csv("naver_keyword_count.csv", encoding="utf-8-sig")
print("\n✅ 저장 완료!")
print(df)


[식당A] 파싱 시작...
[식당A] 카테고리 수집 중 에러 발생: Message: invalid selector: An invalid or illegal selector was specified
  (Session info: chrome=148.0.7778.179); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidselectorexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x91b593+105d3]
	chromedriver!GetHandleVerifier [0x91b6c4+10704]
	chromedriver!(No symbol) [0x721ea0]
	chromedriver!(No symbol) [0x728c1e]
	chromedriver!(No symbol) [0x72ad4a]
	chromedriver!(No symbol) [0x72add8]
	chromedriver!(No symbol) [0x76ad00]
	chromedriver!(No symbol) [0x76b52b]
	chromedriver!(No symbol) [0x7ad052]
	chromedriver!(No symbol) [0x78d7c4]
	chromedriver!(No symbol) [0x7aa936]
	chromedriver!(No symbol) [0x78d516]
	chromedriver!(No symbol) [0x7608e9]
	chromedriver!(No symbol) [0x7616a4]
	chromedriver!GetHandleVerifier [0xba3014+298054]
	chromedriver!GetHandleVerifier [0xb9e603+293643]
	chromedriver!GetHandleVerifier [0xbbea05+2b3a45]


KeyboardInterrupt: 